# Jira Sprint Carryover Report

Analyzes how many issues each team member carried over from one sprint to the next across all closed sprints of a board. Used to evaluate the team KPI: **tasks should not be passed to the next sprint at sprint end**.

## What this notebook produces
1. A per-person × per-sprint matrix (counts)
2. Heatmap visualization of the matrix
3. Trend lines per person
4. **Drill-down: exactly which tickets each person carried, with links**
5. CSV/Excel exports for sharing

## Definitions
- **Carried over** = issue was in the sprint at close AND status was not `Done` → auto-moved to next sprint by Jira
- **Punted** = issue was manually removed from the sprint before it ended (tracked separately; sometimes legitimate scope change)

Adjust `COUNT_PUNTED` in the config cell depending on how your KPI is defined.

## 1. Setup

In [ ]:
# Install if needed (uncomment on first run)
# !pip install requests pandas matplotlib seaborn openpyxl

In [ ]:
import os
import time
from collections import defaultdict
from datetime import datetime

import requests
from requests.auth import HTTPBasicAuth
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML, Markdown

sns.set_style('whitegrid')
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 120)

## 2. Config

Put credentials in env vars so the notebook stays shareable:

```bash
export JIRA_EMAIL="you@granton.ai"
export JIRA_TOKEN="..."   # https://id.atlassian.com/manage-profile/security/api-tokens
export JIRA_BOARD_ID="123"
```

In [ ]:
# ── SECRETS ──────────────────────────────────────────────────
# Loaded from the .env file in THIS project folder (git-ignored).
# First run:  cp .env.example .env  and fill it in.
# Start Jupyter from this folder so the kernel's working directory matches.
import os
from pathlib import Path

from dotenv import load_dotenv

ENV_PATH = Path.cwd() / '.env'
if not ENV_PATH.is_file():
    raise RuntimeError(
        f'No .env found at {ENV_PATH}. Copy .env.example to .env in this '
        f'project folder and fill it in (and start Jupyter from this folder).'
    )
load_dotenv(ENV_PATH)


def env(name, default=None, required=True):
    """Read a setting from this folder's .env / the environment."""
    val = os.getenv(name, default)
    if required and not val:
        raise RuntimeError(
            f'Missing required setting: {name}. Add it to {ENV_PATH} '
            f'(see .env.example).'
        )
    return val

JIRA_DOMAIN = env('JIRA_DOMAIN')
JIRA_EMAIL  = env('JIRA_EMAIL')
JIRA_TOKEN  = env('JIRA_TOKEN')
BOARD_ID    = int(env('JIRA_BOARD_ID'))

# Analysis options
COUNT_PUNTED     = False   # True = include manually-removed issues in carryover
SPRINT_LIMIT     = None    # e.g. 10 to analyze last 10 closed sprints; None = all
REQUEST_DELAY_S  = 0.1     # be polite to the API

BASE    = f'https://{JIRA_DOMAIN}'
AUTH    = HTTPBasicAuth(JIRA_EMAIL, JIRA_TOKEN)
HEADERS = {'Accept': 'application/json'}

assert JIRA_TOKEN and BOARD_ID, 'Set JIRA_TOKEN and JIRA_BOARD_ID env vars'
print(f'Target: {BASE}  board={BOARD_ID}')

## 3. Fetch all closed sprints

In [ ]:
def get_closed_sprints(board_id):
    sprints, start_at = [], 0
    while True:
        r = requests.get(
            f'{BASE}/rest/agile/1.0/board/{board_id}/sprint',
            params={'state': 'closed', 'startAt': start_at, 'maxResults': 50},
            auth=AUTH, headers=HEADERS, timeout=30,
        )
        r.raise_for_status()
        data = r.json()
        sprints.extend(data['values'])
        if data.get('isLast', True):
            break
        start_at += len(data['values'])
        time.sleep(REQUEST_DELAY_S)
    sprints.sort(key=lambda s: s.get('endDate') or '')
    return sprints

YEAR_FILTER        = 2026      # filters by sprint startDate year; set to None to include all
SPRINT_NAME_PREFIX = 'GRT AI'  # set to None to include all sprint names

all_sprints = get_closed_sprints(BOARD_ID)
if YEAR_FILTER:
    all_sprints = [s for s in all_sprints
                   if (s.get('startDate') or '').startswith(str(YEAR_FILTER))]
if SPRINT_NAME_PREFIX:
    all_sprints = [s for s in all_sprints
                   if s.get('name', '').startswith(SPRINT_NAME_PREFIX)]
if SPRINT_LIMIT:
    all_sprints = all_sprints[-SPRINT_LIMIT:]

print(f'Analyzing {len(all_sprints)} closed sprints')
if all_sprints:
    print(f'  from: {all_sprints[0]["name"]}  ({all_sprints[0].get("startDate", "?")} → {all_sprints[0].get("endDate", "?")})')
    print(f'  to:   {all_sprints[-1]["name"]}  ({all_sprints[-1].get("startDate", "?")} → {all_sprints[-1].get("endDate", "?")})')

## 4. Fetch sprint reports (carryover data)

Uses the `greenhopper/sprintreport` endpoint that powers Jira's own Sprint Report UI. Returns:
- `completedIssues` — finished inside the sprint ✅
- `issuesNotCompletedInCurrentSprint` — still in sprint at close, not Done → **auto-carried**
- `puntedIssues` — removed from sprint before it closed
- `issuesCompletedInAnotherSprint` — finished, but credited to a different sprint

In [ ]:
def get_sprint_report(board_id, sprint_id):
    r = requests.get(
        f'{BASE}/rest/greenhopper/1.0/rapid/charts/sprintreport',
        params={'rapidViewId': board_id, 'sprintId': sprint_id},
        auth=AUTH, headers=HEADERS, timeout=30,
    )
    r.raise_for_status()
    return r.json()['contents']


def issue_to_row(issue, sprint, bucket):
    return {
        'sprint_id':    sprint['id'],
        'sprint_name':  sprint['name'],
        'sprint_end':   sprint.get('endDate'),
        'bucket':       bucket,   # 'completed', 'not_completed', or 'punted'
        'issue_key':    issue.get('key'),
        'summary':      issue.get('summary'),
        'assignee':     issue.get('assigneeName') or issue.get('assignee') or 'Unassigned',
        'status':       (issue.get('status') or {}).get('name'),
        'issue_type':   (issue.get('typeName') or issue.get('typeId')),
        'priority':     (issue.get('priorityName') or issue.get('priorityId')),
        'story_points': issue.get('currentEstimateStatistic', {})
                             .get('statFieldValue', {}).get('value'),
        'url':          f'{BASE}/browse/{issue.get("key")}',
    }


rows = []
for i, sp in enumerate(all_sprints, 1):
    try:
        contents = get_sprint_report(BOARD_ID, sp['id'])
    except requests.HTTPError as e:
        print(f'  ! {sp["name"]}: {e}')
        continue

    for issue in contents.get('completedIssues', []):
        rows.append(issue_to_row(issue, sp, 'completed'))
    for issue in contents.get('issuesNotCompletedInCurrentSprint', []):
        rows.append(issue_to_row(issue, sp, 'not_completed'))
    for issue in contents.get('puntedIssues', []):
        rows.append(issue_to_row(issue, sp, 'punted'))

    print(f'  [{i}/{len(all_sprints)}] {sp["name"]}: '
          f'{len(contents.get("completedIssues", []))} done, '
          f'{len(contents.get("issuesNotCompletedInCurrentSprint", []))} not done, '
          f'{len(contents.get("puntedIssues", []))} punted')
    time.sleep(REQUEST_DELAY_S)

df = pd.DataFrame(rows)
df['sprint_end'] = pd.to_datetime(df['sprint_end'], errors='coerce')
df['relevant'] = 1   # set to 0 in Excel to exclude from rate calculations
print(f'\nTotal records: {len(df)}  '
      f'({len(df[df.bucket=="completed"])} done, '
      f'{len(df[df.bucket=="not_completed"])} not done, '
      f'{len(df[df.bucket=="punted"])} punted)')
df.head()

## 5. Build per-person × per-sprint matrix

In [ ]:
if COUNT_PUNTED:
    work_df = df.copy()
else:
    work_df = df[df['bucket'] == 'not_completed'].copy()

sprint_order = [s['name'] for s in all_sprints]

matrix = (work_df
          .groupby(['assignee', 'sprint_name']).size()
          .unstack(fill_value=0)
          .reindex(columns=sprint_order, fill_value=0))

matrix['TOTAL'] = matrix.sum(axis=1)
matrix = matrix.sort_values('TOTAL', ascending=False)

display(Markdown('### Carryover matrix (rows = people, cols = sprints)'))
display(matrix.style.background_gradient(
    cmap='Reds', subset=[c for c in matrix.columns if c != 'TOTAL']
))

## 6. Heatmap

In [ ]:
heatmap_data = matrix.drop(columns=['TOTAL'])

fig_h = max(3, 0.45 * len(heatmap_data))
fig_w = max(8, 0.6 * len(heatmap_data.columns))
fig, ax = plt.subplots(figsize=(fig_w, fig_h))
sns.heatmap(heatmap_data, annot=True, fmt='d', cmap='Reds', linewidths=0.5,
            cbar_kws={'label': 'Issues carried'}, ax=ax)
ax.set_title('Issues carried over per person per sprint')
ax.set_xlabel('Sprint')
ax.set_ylabel('Assignee')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 7. Trend per person

In [ ]:
fig, ax = plt.subplots(figsize=(max(10, 0.55 * len(sprint_order)), 6))

top_n = 10   # avoid a spaghetti chart if team is large
top_people = matrix.head(top_n).index

for person in top_people:
    ax.plot(sprint_order, heatmap_data.loc[person], marker='o', label=person)

ax.set_title(f'Carryover trend (top {min(top_n, len(matrix))} by total)')
ax.set_xlabel('Sprint')
ax.set_ylabel('Issues carried')
ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 8. Team-level trend

Useful context for the KPI — is the team as a whole improving?

In [ ]:
team_trend = work_df.groupby('sprint_name').size().reindex(sprint_order, fill_value=0)

fig, ax = plt.subplots(figsize=(max(10, 0.55 * len(sprint_order)), 4))
ax.bar(team_trend.index, team_trend.values, color='#c0392b', alpha=0.8)
# simple moving average trendline
window = min(3, len(team_trend))
if window >= 2:
    ma = team_trend.rolling(window, min_periods=1).mean()
    ax.plot(ma.index, ma.values, color='black', linewidth=2,
            label=f'{window}-sprint moving avg')
    ax.legend()
ax.set_title('Total issues carried over per sprint (whole team)')
ax.set_xlabel('Sprint')
ax.set_ylabel('Issues carried')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 9. Drill-down: exactly which tickets each person carried

This is what you need for 1:1 conversations — you can go to a person and say "these specific issues spilled over."

In [ ]:
def tickets_for(person, include_punted=COUNT_PUNTED):
    buckets = ['not_completed', 'punted'] if include_punted else ['not_completed']
    d = df[(df['assignee'] == person) & (df['bucket'].isin(buckets))].copy()
    d = d.sort_values(['sprint_end', 'issue_key'])
    return d[['sprint_name', 'bucket', 'issue_key', 'summary', 'status',
              'issue_type', 'priority', 'story_points', 'url']]


def show_person(person):
    tix = tickets_for(person)
    display(Markdown(f'### {person} — {len(tix)} carried issue(s)'))
    if tix.empty:
        print('  (none)')
        return
    # clickable links in the notebook
    html = tix.copy()
    html['issue_key'] = html.apply(
        lambda r: f'<a href="{r.url}" target="_blank">{r.issue_key}</a>', axis=1)
    html = html.drop(columns=['url'])
    display(HTML(html.to_html(escape=False, index=False)))

# Show all people, heaviest carryover first
for person in matrix.index:
    if person == 'Unassigned':
        continue
    show_person(person)

## 10. Inspect a single person

Quick cell for 1:1s — change the name and re-run.

In [ ]:
PERSON = matrix.index[0] if len(matrix) else None   # default = worst offender
if PERSON:
    show_person(PERSON)

## 11. Repeat offenders

Same issue carried across multiple sprints = especially problematic.

In [ ]:
repeat = (work_df.groupby(['issue_key', 'summary', 'assignee'])
                 .agg(times_carried=('sprint_name', 'nunique'),
                      sprints=('sprint_name', lambda x: ', '.join(sorted(set(x)))))
                 .reset_index()
                 .query('times_carried >= 2')
                 .sort_values('times_carried', ascending=False))

display(Markdown(f'### Issues carried across 2+ sprints ({len(repeat)} total)'))
repeat_display = repeat.copy()
repeat_display['issue_key'] = repeat_display['issue_key'].apply(
    lambda k: f'<a href="{BASE}/browse/{k}" target="_blank">{k}</a>')
display(HTML(repeat_display.to_html(escape=False, index=False)))

## 12. Export

In [ ]:
from openpyxl.utils import get_column_letter
from openpyxl.styles import Font, PatternFill, Alignment

stamp = datetime.now().strftime('%Y%m%d_%H%M')

matrix.to_csv(f'carryover_matrix_{stamp}.csv')
df.to_csv(f'carryover_details_{stamp}.csv', index=False)

# Strip timezone — Excel does not support tz-aware datetimes
df_xl = df.copy()
df_xl['sprint_end'] = df_xl['sprint_end'].dt.tz_localize(None)

# Column positions in 'All issues' sheet (index=False, so col A = first df column)
# sprint_id(A) sprint_name(B) sprint_end(C) bucket(D) issue_key(E) summary(F)
# assignee(G) status(H) issue_type(I) priority(J) story_points(K) url(L) relevant(M)
_COL_SPRINT   = 'B'
_COL_BUCKET   = 'D'
_COL_ASSIGNEE = 'G'
_COL_RELEVANT = 'M'

with pd.ExcelWriter(f'carryover_report_{stamp}.xlsx', engine='openpyxl') as xl:
    matrix.to_excel(xl, sheet_name='Matrix')
    df_xl.to_excel(xl, sheet_name='All issues', index=False)
    if not repeat.empty:
        repeat.to_excel(xl, sheet_name='Repeat offenders', index=False)
    for person in matrix.index:
        if person == 'Unassigned':
            continue
        sheet = person[:31].replace('/', '_')
        tickets_for(person).to_excel(xl, sheet_name=sheet, index=False)

    # --- Rate sheet: carryover % per person per sprint (respects relevant=1 filter) ---
    persons = [p for p in matrix.index if p != 'Unassigned']
    sprints = [s['name'] for s in all_sprints]
    ws = xl.book.create_sheet('Rate %')

    header_fill = PatternFill('solid', fgColor='2F4F8F')
    header_font = Font(bold=True, color='FFFFFF')

    # Header row: sprint names
    ws.cell(1, 1, 'Assignee').font = Font(bold=True)
    for ci, sprint in enumerate(sprints, 2):
        c = ws.cell(1, ci, sprint)
        c.fill = header_fill
        c.font = header_font
        c.alignment = Alignment(horizontal='center')

    for ri, person in enumerate(persons, 2):
        ws.cell(ri, 1, person).font = Font(bold=True)
        for ci, sprint in enumerate(sprints, 2):
            p_ref = f"$A{ri}"
            s_ref = f"{get_column_letter(ci)}$1"
            carried = (f"COUNTIFS('All issues'!${_COL_ASSIGNEE}:${_COL_ASSIGNEE},{p_ref},"
                       f"'All issues'!${_COL_SPRINT}:${_COL_SPRINT},{s_ref},"
                       f"'All issues'!${_COL_BUCKET}:${_COL_BUCKET},\"not_completed\","
                       f"'All issues'!${_COL_RELEVANT}:${_COL_RELEVANT},1)")
            total   = (f"COUNTIFS('All issues'!${_COL_ASSIGNEE}:${_COL_ASSIGNEE},{p_ref},"
                       f"'All issues'!${_COL_SPRINT}:${_COL_SPRINT},{s_ref},"
                       f"'All issues'!${_COL_RELEVANT}:${_COL_RELEVANT},1)")
            cell = ws.cell(ri, ci)
            cell.value = f'=IF({total}=0,"",{carried}/{total})'
            cell.number_format = '0%'
            cell.alignment = Alignment(horizontal='center')

    ws.column_dimensions['A'].width = 22

print(f'Wrote:')
print(f'  carryover_matrix_{stamp}.csv')
print(f'  carryover_details_{stamp}.csv')
print(f'  carryover_report_{stamp}.xlsx  (sheets: Matrix, All issues, Rate %, per-person, Repeat offenders)')